In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, ops
import matplotlib.pyplot as plt
import os
import numpy as np

In [ ]:
tf.config.list_physical_devices('GPU')

In [ ]:
labels = [images.split('.')[0] for images in os.listdir('/kaggle/input/captcha-images/captcha_images')]

In [ ]:
characters = set(char for label in labels for char in label)
characters = sorted(list(characters))

In [ ]:
char_to_num = layers.StringLookup(vocabulary=list(characters), mask_token=None)

num_to_char = layers.StringLookup(
    vocabulary=char_to_num.get_vocabulary(), mask_token=None, invert=True
)

In [ ]:
labels = char_to_num(tf.strings.unicode_split(labels, input_encoding="UTF-8"))

In [ ]:
train_images = [images for images in os.listdir('/kaggle/input/captcha-images/captcha_images')]

In [ ]:
directory = '/kaggle/input/captcha-images/captcha_images'

def preprocess_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.io.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [32, 32])
    img = tf.transpose(img, perm=[1, 0, 2])
    return img

image_paths = [os.path.join(directory, fname) for fname in os.listdir(directory)]

processed_images = [preprocess_image(path) for path in image_paths]

batch = tf.stack(processed_images)  # Shape: (num_images, width, height, channels).

In [ ]:
labels = [[label] for label in labels]

In [ ]:
labels = tf.reshape(labels, shape=(9955, 4))

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((batch, labels))

In [ ]:
buffer = tf.data.AUTOTUNE
dataset = dataset.batch(32).prefetch(buffer_size=buffer).shuffle(buffer_size=dataset.cardinality())

In [ ]:
_, ax = plt.subplots(4, 4, figsize=(10, 5))
for images, labels in dataset.take(1):
    for i in range(16):
        img = (images[i] * 255).numpy().astype("uint8")
        label = tf.strings.reduce_join(num_to_char(labels[i])).numpy().decode("utf-8")
        ax[i // 4, i % 4].imshow(img[:, :, 0].T, cmap="gray")
        ax[i // 4, i % 4].set_title(label)
        ax[i // 4, i % 4].axis("off")
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Shape: height, width, channels.
input_shape = (32, 32, 1)
input_image = layers.Input(shape=input_shape)

x = layers.Conv2D(32, (3, 3), activation='relu', padding='same')(input_image)
x = layers.MaxPooling2D((2, 2))(x)  # Downsample spatial dimensions.
x = layers.Conv2D(64, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)
x = layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
x = layers.MaxPooling2D((2, 2))(x)

# Reshape CNN features for the RNN.
x = layers.Reshape(target_shape=(-1, x.shape[-1]))(x)
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=.25))(x)
x = layers.Bidirectional(layers.LSTM(128, return_sequences=True, dropout=.25))(x)

x = layers.Dense(len(char_to_num.get_vocabulary()) + 1)(x)

model = models.Model(input_image, x)

model.summary()

In [ ]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), loss='CTC', metrics=['accuracy'])

In [ ]:
model.fit(dataset, epochs=100)

In [ ]:
directory = '/kaggle/input/captcha-images/captcha_images'

def preprocess_image(image_path):
    img = tf.io.read_file(image_path)
    img = tf.io.decode_png(img, channels=1)
    img = tf.image.convert_image_dtype(img, tf.float32)
    img = tf.image.resize(img, [32, 32])
    img = tf.transpose(img, perm=[1, 0, 2])
    return img

image_paths = [os.path.join(directory, fname) for fname in os.listdir(directory)]

test_images = [preprocess_image(path) for path in image_paths]

test_images = tf.stack(test_images)  # Shape: (num_images, width, height, channels).
print(f"Batch shape: {test_images.shape}")

In [ ]:
labels = [images.split('.')[0] for images in os.listdir('/kaggle/input/captcha-images/captcha_images')]

In [ ]:
labels = char_to_num(tf.strings.unicode_split(labels, input_encoding="UTF-8"))

In [ ]:
labels = [[label] for label in labels]

In [ ]:
labels = tf.reshape(labels, shape=(9955, 4))

In [ ]:
dataset = tf.data.Dataset.from_tensor_slices((test_images, labels))

In [ ]:
dataset = dataset.batch(32).prefetch(buffer_size=buffer)

In [ ]:
def ctc_decode(y_pred, input_length, greedy=True, beam_width=100, top_paths=1):
    input_shape = ops.shape(y_pred)
    num_samples, num_steps = input_shape[0], input_shape[1]
    y_pred = ops.log(ops.transpose(y_pred, axes=[1, 0, 2]) + keras.backend.epsilon())
    input_length = ops.cast(input_length, dtype="int32")

    if greedy:
        (decoded, log_prob) = tf.nn.ctc_greedy_decoder(
            inputs=y_pred, sequence_length=input_length
        )
    else:
        (decoded, log_prob) = tf.compat.v1.nn.ctc_beam_search_decoder(
            inputs=y_pred,
            sequence_length=input_length,
            beam_width=beam_width,
            top_paths=top_paths,
        )
    decoded_dense = []
    for st in decoded:
        st = tf.SparseTensor(st.indices, st.values, (num_samples, num_steps))
        decoded_dense.append(tf.sparse.to_dense(sp_input=st, default_value=-1))
    return (decoded_dense, log_prob)

prediction_model = keras.models.Model(
    inputs=input_image, outputs=x
)

def decode_batch_predictions(pred, max_length=4):
    input_len = np.ones(pred.shape[0]) * pred.shape[1]
    # Greedy CTC decoding; beam search is also available.
    results = ctc_decode(pred, input_length=input_len, greedy=True)[0][0][
        :, :max_length
    ]
    output_text = []
    for res in results:
        res = tf.strings.reduce_join(num_to_char(res)).numpy().decode("utf-8")
        output_text.append(res)
    return output_text

for images, labels in dataset.take(1):

    preds = prediction_model.predict(images)
    pred_texts = decode_batch_predictions(preds)

    orig_texts = []
    for label in labels:
        label = tf.strings.reduce_join(num_to_char(label)).numpy().decode("utf-8")
        orig_texts.append(label)

    _, ax = plt.subplots(4, 4, figsize=(15, 5))
    ax = ax.flatten()
    num_to_plot = min(len(pred_texts), len(ax))
    for i in range(num_to_plot):
        img = (images[i, :, :, 0] * 255).numpy().astype(np.uint8)
        img = img.T
        title = f"Prediction: {pred_texts[i]}"
        ax[i].imshow(img, cmap="gray")
        ax[i].set_title(title)
        ax[i].axis("off")

plt.tight_layout()
plt.show()